# 01 · ¿Qué es MCP?

**MCP (Model Context Protocol)** es un protocolo abierto (creado por Anthropic, ahora
con adopción multi-vendor) que estandariza **cómo una aplicación le da a un modelo de
lenguaje acceso a herramientas, datos y prompts externos**.

Antes de MCP, cada framework de agentes tenía su propia forma de definir "tools":
LangChain con `@tool`, OpenAI con su schema de function-calling, cada SDK con su
convención. Si querías reutilizar la misma herramienta (por ejemplo "consultar el
catálogo de productos de jub") en dos agentes distintos, tenías que reimplementarla dos
veces.

MCP resuelve esto separando **quién implementa la herramienta** de **quién la usa**:
un **servidor MCP** implementa las tools una sola vez; cualquier **cliente MCP**
(cualquier agente, en cualquier framework, en cualquier lenguaje) se conecta y las usa
tal cual. Es literalmente lo mismo que resolvió el *Language Server Protocol* (LSP)
para editores de código: en vez de que cada editor reimplemente el análisis de cada
lenguaje, el análisis vive en un servidor y cualquier editor lo consume.

En este notebook vas a entender el protocolo por dentro, sin ningún framework encima.


## Arquitectura: host, client, server

MCP define tres roles:

```
┌─────────────────────────────────────────────────────────────┐
│  HOST  (la aplicación: un agente, un IDE, un chat)           │
│                                                               │
│   ┌────────────┐        MCP (JSON-RPC 2.0)     ┌───────────┐│
│   │ MCP CLIENT │ ───────────────────────────── │ MCP SERVER││
│   │ (1 por     │  stdio / HTTP+SSE / WebSocket  │ (expone   ││
│   │  servidor) │ ◀───────────────────────────── │  tools)   ││
│   └────────────┘                                └───────────┘│
└─────────────────────────────────────────────────────────────┘
```

- **Host**: la aplicación que orquesta uno o más agentes (en nuestro caso, el proceso
  de `agent-framework`).
- **Client**: vive dentro del host, mantiene una conexión 1:1 con un servidor MCP.
- **Server**: un proceso separado (o remoto) que expone *tools*, *resources* y
  *prompts*. No sabe nada del LLM ni del host — solo habla el protocolo.

La conexión transporta mensajes **JSON-RPC 2.0** (el mismo formato que usa LSP). Puede
viajar por **stdio** (el server es un subproceso local, típico para tools que corren en
tu máquina), **HTTP con streaming** (`streamable-http`, para servidores persistentes en
red — lo que usa `jub-agent`), o **WebSocket**.


## Las tres primitivas de MCP

| Primitiva  | Qué es                                            | Quién la controla        |
|------------|----------------------------------------------------|---------------------------|
| **tools**    | Funciones que el modelo puede *invocar* (con efectos: escribir, consultar, calcular) | El modelo decide llamarlas |
| **resources**| Datos que el host puede *leer* y meter al contexto (archivos, filas de una BD, docs) | La aplicación decide exponerlas |
| **prompts**  | Plantillas de prompt reutilizables, parametrizadas, que el servidor sugiere         | El usuario/host las invoca |

En este tutorial (y en `jub-agent`) usamos casi exclusivamente **tools**, que son la
primitiva más parecida al "function calling" que ya conoces de OpenAI/Anthropic — la
diferencia es que la tool vive en un servidor MCP independiente, no hardcodeada en el
código del agente.


## Manos a la obra: hablar con un servidor MCP sin ningún framework de agentes

Vamos a:
1. Escribir un servidor MCP mínimo (una sola tool) usando `mcp.server.fastmcp.FastMCP`
   — la clase `FastMCP` que viene **incluida en el SDK oficial `mcp`** (no confundir
   con el paquete de terceros `fastmcp` que usaremos desde el notebook 02 en adelante,
   que es una versión extendida de la misma idea).
2. Lanzarlo como subproceso.
3. Conectarnos con `ClientSession` + `stdio_client` del SDK oficial, listar sus tools y
   llamarlas — exactamente lo que hace `agent-framework` por debajo, pero a mano.

Requisito: `pip install mcp`.


In [2]:
import sys
import tempfile
import textwrap
from pathlib import Path

server_code = textwrap.dedent('''
    from mcp.server.fastmcp import FastMCP

    mcp = FastMCP("mini-server")

    @mcp.tool()
    def saludar(nombre: str) -> str:
        'Devuelve un saludo personalizado para nombre.'
        return f"¡Hola, {nombre}! Este saludo vino de un servidor MCP real."

    if __name__ == "__main__":
        mcp.run()  # transporte stdio por defecto
''')

server_path = Path(tempfile.gettempdir()) / "mini_mcp_server.py"
server_path.write_text(server_code, encoding="utf-8")
print("Servidor escrito en:", server_path)


Servidor escrito en: /tmp/mini_mcp_server.py


In [3]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def hablar_con_el_server():
    params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Tools que expone el servidor:", [t.name for t in tools.tools])
            print("Descripción de 'saludar':", tools.tools[0].description)

            resultado = await session.call_tool("saludar", {"nombre": "Juan"})
            print("Resultado de invocar la tool:", resultado.content[0].text)

await hablar_con_el_server()


Tools que expone el servidor: ['saludar']
Descripción de 'saludar': Devuelve un saludo personalizado para nombre.
Resultado de invocar la tool: ¡Hola, Juan! Este saludo vino de un servidor MCP real.


### Qué acaba de pasar

1. `stdio_client` lanzó `python mini_mcp_server.py` como subproceso y conectó su
   stdin/stdout como transporte.
2. `ClientSession.initialize()` hizo el *handshake* MCP (intercambio de capacidades).
3. `list_tools()` le pidió al servidor su catálogo de tools — el servidor respondió con
   el nombre, la descripción (¡el docstring!) y el JSON Schema de los parámetros de
   `saludar`, generado automáticamente a partir de la firma de la función.
4. `call_tool(...)` envió una petición JSON-RPC `tools/call`, el servidor ejecutó la
   función Python real y devolvió el resultado.

Nada de esto mencionó un LLM. **MCP es solo el protocolo de transporte de tools** — es
un framework de agentes (como `agent-framework`, que vemos en el notebook 03) el que
decide *cuándo* llamar una tool, basándose en lo que el modelo responde.

### De stdio a streamable-http

Para un servidor que va a vivir corriendo de forma persistente en un contenedor y
atender a más de un cliente a la vez (nuestro caso, y el de `jub-agent`), stdio no
alcanza — necesitamos un servidor que escuche en un puerto de red. Ese es exactamente
el rol del transporte `streamable-http`, que usaremos desde el próximo notebook con el
paquete `fastmcp`.

**Siguiente:** [`02_servidor_mcp_basico.ipynb`](02_servidor_mcp_basico.ipynb)
